## 库导入和数据加载

In [ ]:
import numpy as np
import pandas as pd
import pynapple as nap
import matplotlib.pyplot as plt
import tqdm
from joblib import Parallel, delayed
import os

from scipy.interpolate import interp1d
from pathlib import Path
import Output
import Compute
import SpatialProperty


#加载数据
source = Path(r"E:\YLK\Stability\0720")
calcium_path = source / "outputs"/ "calcium.nwb"  # Suite2p 文件夹, 即plane0的上级目录或含有calcium.nwb的文件夹
behavior_path = source / "outputs"/ "behavior.nwb" #行为学数据.nwb
PlaceCell = np.load(source / "outputs" / "place_cells_append.npy")
behavior_data = nap.load_file(behavior_path)
calcium_data = nap.load_file(calcium_path)
Output.print_to_log(calcium_data)
Output.print_to_log(behavior_data)

#提取数据
position = behavior_data['SpatialSeries']
epochs = behavior_data['Running_laps']
lick = behavior_data['licking']
transients = calcium_data["Deconvolved"]

PlaceCell_index = np.where(PlaceCell[:,0]==1)[0]#write_iscell 函数的逆向

save_path = source / 'outputs' / 'property'
os.makedirs(save_path, exist_ok=True)

#### 可视化

In [ ]:
Output.print_to_log(f"position data shape is {position.shape}")
Output.print_to_log(f"transients data shape is {transients.shape}; max value is {np.max(transients)}; at {np.where(transients==np.max(transients))}")
Output.print_to_log(f"laps data shape is {epochs.shape}")
Output.print_to_log(f"PlaceCells is {PlaceCell_index},length is {len(PlaceCell_index)}")

Output.print_to_log("\nTransient of first Cell")
plt.figure(figsize=(10,2))
plt.plot(transients[1:36000,0], linewidth=0.5)
plt.show()

## 预处理

#### 插帧

In [ ]:
# 参数设置
chunk_size = 3600          # 每个块3600帧
insert_frames = 3          # 每个块前插入3帧
n_chunks = 10              # 总块数 (36000/3600=10)
total_inserts = insert_frames * n_chunks  # 总插入30帧
frames_size = chunk_size*n_chunks + total_inserts
new_transients= Compute.frameinsert(chunk_size,n_chunks,insert_frames,total_inserts,transients)


##### 可视化

In [ ]:

# ===== 验证 =====
print(f"插入帧数：{total_inserts} ,总帧数{new_transients.shape[0]}")
plt.figure(figsize=(10,2))
plt.plot(new_transients[1:frames_size,0], linewidth=0.5)
plt.show()

#### 计算放电率

分割化、速度计算、非静止区计算

In [ ]:
track_length = 400
frame_rate = 30
bin_num = 80
min_speed_threshold = 5  # cm.s-1 速度阈值

filtered_transients = new_transients[:,PlaceCell_index]

#构造插值函数并且进行插值
interp_func = interp1d(
    position.times(),
    position.values,
    fill_value=(position.values[0], position.values[-1]),
    bounds_error=False
)
tmp = interp_func(filtered_transients.times())   # use interpolation function returned by `interp1d`
Filtered_Position = nap.Tsd(
    t=filtered_transients.times(),
    d=tmp,
    time_units='s'
)

#分割化
bin_indices = Compute.binning(Filtered_Position,bin_num)
#计算tuningcurve
#这个函数以后可以再拆成单独的三个，现在这样只是简单打包，太丑陋。
#smoothcurves是一个PandasDF,80bin*cells
velocity,is_running =  Compute.compute_velocity(
    epochs,position,track_length,filtered_transients,min_speed_threshold=5)

running_transients = filtered_transients[is_running]
running_position = Filtered_Position[is_running]


##### 可视化

In [ ]:
slice_no = 1
output_path = Path('E:\Szbl\Programming\SmallTools\CellStimAuto\outputs')
Output.Movementfig(filtered_transients,Filtered_Position,velocity,is_running,slice_no,output_path,epochs,lick,position)

## 计算稳定性

#### 计算皮尔森相关系数

这里还有优化空间，先算完单个lap再算单个细胞。现在有大量重复计算

In [ ]:
from contextlib import contextmanager
import multiprocessing

single_tuningcurves = []
for lap in epochs:
    start = lap.start
    end = lap.end
    
    # 截取时间段数据
    timepart = Compute.get_parts(running_transients, end, True)
    timepart = Compute.get_parts(timepart, start, False)
    pospart = Compute.get_parts(running_position, end, True)
    pospart = Compute.get_parts(pospart, start, False)
    singlecurve = Compute.compute_tuningcurves(timepart, pospart, bin_num)
    single_tuningcurves.append(singlecurve)
    

@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_cb = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_cb
        tqdm_object.close()
        
def compute_single_cell_correlation(index, mean_tuningcurve,single_tuningcurves,epochs):
    single_correlations = []
    for singlecurves in single_tuningcurves:
        if True:
            # 计算皮尔逊相关
            corr = Compute.pearson_correlation(mean_tuningcurve[index].values, 
                                                singlecurves[index].values)
            single_correlations.append(corr)
        # 返回该神经元的平均相关性
    return np.mean(single_correlations) if single_correlations else np.nan


# 主计算逻辑
if __name__ == "__main__":
    # 1. 先计算平均调谐曲线（这部分无法并行）
    mean_tuningcurve = Compute.compute_tuningcurves(running_transients, running_position, bin_num)
    
    # 2. 设置并行计算参数
    n_jobs = multiprocessing.cpu_count() - 1  # 使用CPU核心数-1，保留一个核心
    # 如果数据量小，也可以手动设置，比如 n_jobs = 4
    
    # 3. 并行执行计算
    # 使用tqdm显示并行进度
    correlation = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(compute_single_cell_correlation)(
            index, mean_tuningcurve,single_tuningcurves,epochs
            ) for index in tqdm.tqdm(PlaceCell_index, desc="计算神经元稳定性"))

stability = np.array(correlation)

#### 可视化

In [ ]:
Output.compute_distribution(stability,'Stability',save_path)

## 位置场数量

In [ ]:
PlaceCell_tuningcurves = Compute.compute_tuningcurves(running_transients,running_position,bin_num)

Place_field_numbers = []
for index in PlaceCell_index:
    placefields =  SpatialProperty.place_field_identification(
        neuron_idx=index,
        tuning_curves=PlaceCell_tuningcurves,
        transients=new_transients,
        Filtered_Position=running_position,
        epochs=epochs,
        bin_indices=bin_indices,
        FilterField_thresholdRatio=0.25,
        FilterMean_threshold=7,
        FilterTime_thresholdRatio=0.33)
    Place_field_numbers.append(len(placefields))
Place_field_numbers = np.array(Place_field_numbers)
Output.compute_distribution(Place_field_numbers,'placefield number',save_path,0,5,0,1000,False,True)

## 位置场宽度

In [ ]:
PlaceCell_tuningcurves = Compute.compute_tuningcurves(running_transients,running_position,bin_num)

Placefield_width = []
for index in PlaceCell_index:
    placefields =  SpatialProperty.place_field_identification(
        neuron_idx=index,
        tuning_curves=PlaceCell_tuningcurves,
        transients=new_transients,
        Filtered_Position=running_position,
        epochs=epochs,
        bin_indices=bin_indices,
        FilterField_thresholdRatio=0.25,
        FilterMean_threshold=7,
        FilterTime_thresholdRatio=0.33)
    Placefield_width.append(np.mean(placefields))
    
Placefield_width = np.array(Placefield_width)
Output.compute_distribution(Placefield_width,'placefield width(bin_num)',save_path,0,35,0,1000,True,True)


## 空间信息

In [ ]:
PlaceCell_tuningcurves = Compute.compute_tuningcurves(running_transients,Filtered_Position,bin_num)

Spatial_info = []
for i in range(len(PlaceCell_index)):
    activity = filtered_transients[:,i]
    index = PlaceCell_index[i]
    Spatial_info.append(SpatialProperty.compute_spatial_information(activity=activity,neuron_idx=index,tuning_curves=PlaceCell_tuningcurves,bin_indices=bin_indices))
    
Spatial_info = np.array(Spatial_info)
Output.compute_distribution(Spatial_info,'spatialinfo(bit_per_spike)',save_path,-1,7,0,1000,True,True)


## 活跃度

#### mean_dff

In [ ]:
mean_dffs = []
for i in range(len(PlaceCell_index)):
    mean_dff = filtered_transients[:,i].mean()
    mean_dffs.append(mean_dff)
    
mean_dffs = np.array(mean_dffs)
max_diff = mean_dffs.max()
Output.compute_distribution(mean_dffs,'mean_dff',save_path,0,max_diff+0.01,0,200,True,True)

## 保存

In [ ]:
save = pd.DataFrame({'stability':stability,"PFnumbers":Place_field_numbers,"PFwidth":Placefield_width,"Spatialinfo":Spatial_info,"meandff":mean_dffs},index=PlaceCell_index)
save.to_csv(save_path/'results.csv')
